# Create Helse Sør-Øst Awards (Helse Sør-Øst RHF regional research funds)

Creates South-Eastern Norway Regional Health Authority (Helse Sør-Øst RHF)
research awards from the authority's own consolidated allocation list,
"Tildelinger - regionale forskningsmidler 2011 til 2023" (105-page PDF,
https://www.helse-sorost.no/4aee52/contentassets/7c84e81a540f46809f6b17cb87483707/tildelinger---regionale-forskningsmidler-2011-til-2023.pdf).
One row per funded project: project leader, hospital, title, application
type (doctoral / postdoctoral / researcher / career fellowship, open project
support, research group, network, core facility, ...), duration, and the
amount allocated for the award year. **1,408 projects, 2011-2023, 100%
title/leader/hospital, 98.5% amount** (local run 2026-10-01).

All rows are medical/health research funding; nothing is filtered out.

**Parsing (in `scripts/local/helse_sor_ost_to_s3.py`):** ~15 table layouts
across the years; columns are mapped by header name, the hospital comes from
the group-header row or the line printed above each table, duplicated glyphs
on the 2012 pages are de-duplicated, and 2021-2023 amounts printed in NOK
thousands are scaled.

**Amount semantics:** `amount` = the allocation printed for the award year
(the project's first-year amount, NOK), which is what the lists publish.
Multi-year totals are printed only for 2021 (kept as `amount_total` in the
raw table). 21 rows have no amount in the source (2015 core facilities,
2022 non-university-hospital strategic funds, 2022 career grants).

**Dates:** `start_year` = award year; `start_date` from the printed start
date where given (2019-2022 lists, 27%); `end_date` = start + duration where
both are known.

**`funder_award_id` (2.1.1):** the lists print no project number. Citing
works write Helse Sør-Øst's 7-digit project numbers (e.g. `2017096`; ~1,970
of ~3,230 acknowledgement shell rows), which are not in this source, so
those shells cannot be collapsed. Synthetic key
`HSO-<year>-<leader slug>-<sha1(title)[:6]>`. Zero existing shells use it.

**Not ingested (follow-up):** 2024+ allocations (only in later board
papers). The four regional health authorities share a national project
register (forskningsprosjekter.ihelse.net, "eRapport"), but its robots.txt
disallows the report pages and its search backend, so it was not used.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320335102`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 02qx2s478, doi 10.13039/501100006095)

**Priority:** `527` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.helse_sor_ost_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/helse_sor_ost/helse_sor_ost_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.helse_sor_ost_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.helse_sor_ost_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.helse_sor_ost_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320335102 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320335102;

## Step 2: Create Helse Sør-Øst Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.helse_sor_ost_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320335102  -- Helse Sør-Øst RHF
),

parsed AS (
    SELECT g.*,
           TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') AS start_d,
           TRY_CAST(g.duration_months AS INT) AS months
    FROM openalex.awards.helse_sor_ost_raw g
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount_year AS DOUBLE) as amount,
    CASE WHEN g.amount_year IS NOT NULL THEN 'NOK' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.funder_scheme RLIKE '(?i)stipend' THEN 'fellowship' ELSE 'research' END as funding_type,
    g.funder_scheme as funder_scheme,
    'helse_sor_ost_tildelinger' as provenance,
    g.start_d as start_date,
    CASE WHEN g.start_d IS NOT NULL AND g.months IS NOT NULL
         THEN date_add(add_months(g.start_d, g.months), -1) END as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CASE WHEN g.start_d IS NOT NULL AND g.months IS NOT NULL
         THEN YEAR(date_add(add_months(g.start_d, g.months), -1)) END as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'NO',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    'https://www.helse-sorost.no/4aee52/contentassets/7c84e81a540f46809f6b17cb87483707/tildelinger---regionale-forskningsmidler-2011-til-2023.pdf' as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.helse_sor_ost_awards
WHERE NOT funder_award_id RLIKE '^HSO-(19|20)[0-9]{2}-[a-z0-9-]+-[0-9a-f]{6}$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.helse_sor_ost_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'helse_sor_ost_tildelinger' AND priority = 527;

-- Priority 527: direct-from-funder ingest of Helse Sør-Øst's own allocation list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    527 as priority
FROM openalex.awards.helse_sor_ost_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.helse_sor_ost_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_nok
FROM openalex.awards.helse_sor_ost_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_nok
FROM openalex.awards.helse_sor_ost_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.helse_sor_ost_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.helse_sor_ost_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'helse_sor_ost_tildelinger'
GROUP BY provenance, priority;